In [50]:
from script.OCR import ocr
import cv2
from ultralytics import YOLO
import glob
import os
import numpy as np

In [3]:
text_ocr = ocr()

g:\Python\MVP\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


**YOLO Model (Validation Data)**

In [4]:
model = YOLO("models/best.pt")

test_images = glob.glob("test_images/*.jpg") + glob.glob("test_images/*.png")
print(f"Found {len(test_images)} test images")

results = model.predict(
    source=test_images,
    save=True,
    project=r"G:\Python\MVP\outputs",
    name="predict",
    exist_ok=True,
    conf=0.5
)

Found 30 test images

0: 640x640 5 0s, 1150.1ms
1: 640x640 5 0s, 1150.1ms
2: 640x640 1 0, 1150.1ms
3: 640x640 4 0s, 1150.1ms
4: 640x640 4 0s, 1150.1ms
5: 640x640 1 0, 1150.1ms
6: 640x640 10 0s, 1150.1ms
7: 640x640 1 0, 1150.1ms
8: 640x640 10 0s, 1150.1ms
9: 640x640 9 0s, 1150.1ms
10: 640x640 6 0s, 1150.1ms
11: 640x640 11 0s, 1150.1ms
12: 640x640 6 0s, 1150.1ms
13: 640x640 21 0s, 1150.1ms
14: 640x640 23 0s, 1150.1ms
15: 640x640 21 0s, 1150.1ms
16: 640x640 19 0s, 1150.1ms
17: 640x640 20 0s, 1150.1ms
18: 640x640 27 0s, 1150.1ms
19: 640x640 8 0s, 1150.1ms
20: 640x640 17 0s, 1150.1ms
21: 640x640 14 0s, 1150.1ms
22: 640x640 15 0s, 1150.1ms
23: 640x640 10 0s, 1150.1ms
24: 640x640 10 0s, 1150.1ms
25: 640x640 14 0s, 1150.1ms
26: 640x640 14 0s, 1150.1ms
27: 640x640 16 0s, 1150.1ms
28: 640x640 29 0s, 1150.1ms
29: 640x640 5 0s, 1150.1ms
Speed: 18.5ms preprocess, 1150.1ms inference, 29.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to G:\Python\MVP\outputs\predict


**Image Croping**

In [5]:
output_dir = r"G:\Python\MVP\outputs\crops"
os.makedirs(output_dir, exist_ok=True)

for img_path, result in zip(test_images, results):
    img = cv2.imread(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]

    boxes = result.boxes.xyxy.cpu().numpy()
    h, w = img.shape[:2]

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = img[y1: y2, x1: x2]

        crop_filename = os.path.join(output_dir, f"{img_name}_spine{i}.jpg")
        cv2.imwrite(crop_filename, crop)

**YOLO Model (Testing)**

In [6]:
model = YOLO("models/best.pt")

test_images = glob.glob("test_images_2/*.jfif") + glob.glob("test_images_2/*.jpg")
print(f"Found {len(test_images)} test images")

results = model.predict(
    source=test_images,
    save=True,
    project=r"G:\Python\MVP\outputs_2",
    name="predict",
    exist_ok=True,
    conf=0.5
)

Found 5 test images

0: 640x640 20 0s, 179.5ms
1: 640x640 19 0s, 179.5ms
2: 640x640 7 0s, 179.5ms
3: 640x640 8 0s, 179.5ms
4: 640x640 16 0s, 179.5ms
Speed: 53.5ms preprocess, 179.5ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)
Results saved to G:\Python\MVP\outputs_2\predict


**Cropping**

In [7]:
output_dir = r"G:\Python\MVP\outputs_2\crops"
os.makedirs(output_dir, exist_ok=True)

for img_path, result in zip(test_images, results):
    img = cv2.imread(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]

    boxes = result.boxes.xyxy.cpu().numpy()
    h, w = img.shape[:2]

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = img[y1: y2, x1: x2]

        crop_filename = os.path.join(output_dir, f"{img_name}_spine{i}.jpg")
        cv2.imwrite(crop_filename, crop)

**Gray Scaling**

In [28]:
os.makedirs('outputs_2/gray', exist_ok=True)
os.makedirs('outputs_2/clahe', exist_ok=True)

crop_paths = glob.glob('outputs_2/crops/*.jpg')
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

for path in crop_paths:
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    clahe_gray = clahe.apply(gray)

    filename = os.path.basename(path)
    cv2.imwrite(f'outputs_2/gray/{filename}', gray)
    cv2.imwrite(f'outputs_2/clahe/{filename}', clahe_gray)

In [47]:
img = cv2.imread("outputs_2/crops/images_spine5.jpg")
rotated = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
scale = 4

rotated = cv2.resize(
    rotated,
    None,
    fx=scale,
    fy=scale,
    interpolation=cv2.INTER_CUBIC
)

# results = text_ocr.reader.readtext(rotated)
results = text_ocr.txt(rotated)
print(results)

Text: siqnog QiuoseW Nxeem
Confidence: 0.087
Text: 4ienjoh ueaisi
Confidence: 0.265
None


In [51]:
os.makedirs("outputs_2/preprocessed", exist_ok=True)
os.makedirs("outputs_2/binary", exist_ok=True)

crop_paths = glob.glob("outputs_2/crops/*.jpg")

clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))

for path in crop_paths:

    img = cv2.imread(path)

    # Skip unreadable images
    if img is None:
        continue

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Upscale
    scale = 4
    gray = cv2.resize(
        gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    # Denoise
    gray = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)

    # CLAHE
    gray = clahe.apply(gray)

    # Sharpen
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    sharpen = cv2.filter2D(
        gray,
        -1,
        kernel=np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])
    )

    # Otsu Threshold
    _, binary = cv2.threshold(
        sharpen,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    filename = os.path.basename(path)

    cv2.imwrite(
        os.path.join("outputs_2/preprocessed", filename),
        sharpen
    )

    cv2.imwrite(
        os.path.join("outputs_2/binary", filename),
        binary
    )

print("Done!")

Done!


In [ ]:
import cv2

img = cv2.imread("outputs_2/binary/images_spine5.jpg", cv2.IMREAD_GRAYSCALE)

results = text_ocr.reader.readtext(img)

print(results)